# 09. Markov Grid Localization

Grid localization은 상태공간을 격자로 이산화하고 각 cell의 확률을 직접 유지한다.

$$bel(x_t)=\eta p(z_t\mid x_t)\sum_{x_{t-1}}p(x_t\mid u_t,x_{t-1})bel(x_{t-1})$$

Particle filter보다 계산량은 크지만, 작은 지도에서는 belief 전체가 어떻게 이동하고 날카로워지는지 보기 좋다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 2D Grid Bayes Filter

로봇은 격자 지도 안에서 상하좌우로 움직인다. 센서는 현재 cell 주변의 벽 개수만 노이즈 있게 알려준다.

In [ ]:
np.random.seed(21)
H,W=12,16
occ=np.zeros((H,W),dtype=bool)
occ[0,:]=occ[-1,:]=occ[:,0]=occ[:,-1]=True
occ[3:9,5]=True; occ[6,8:13]=True; occ[2:5,11]=True
free=np.argwhere(~occ)
bel=np.where(~occ,1.0,0.0); bel/=bel.sum()
true=(9,2)
cmds=[(0,1),(0,1),(-1,0),(-1,0),(0,1),(0,1),(0,1),(1,0),(0,1)]

def wall_count(s):
    r,c=s; cnt=0
    for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        cnt += occ[r+dr,c+dc]
    return cnt

def move_state(s,u):
    ns=(s[0]+u[0],s[1]+u[1])
    return s if occ[ns] else ns

def predict_grid(bel,u):
    out=np.zeros_like(bel)
    motions=[(u,0.78),((0,0),0.12),((-u[1],u[0]),0.05),((u[1],-u[0]),0.05)]
    for r,c in free:
        for du,p in motions:
            ns=move_state((r,c),du)
            out[ns]+=p*bel[r,c]
    return out/out.sum()

def correct_grid(bel,z,sigma=0.65):
    like=np.zeros_like(bel)
    for r,c in free:
        like[r,c]=np.exp(-0.5*((z-wall_count((r,c)))/sigma)**2)
    out=bel*like
    return out/out.sum()

frames=[]
for u in cmds:
    true=move_state(true,u)
    z=wall_count(true)+np.random.randn()*0.45
    bel=predict_grid(bel,u)
    bel=correct_grid(bel,z)
    frames.append((bel.copy(),true,z))

fig,axes=plt.subplots(3,3,figsize=(12,9),sharex=True,sharey=True)
for ax,(b,t,z),i in zip(axes.ravel(),frames,range(1,len(frames)+1)):
    show=b.copy(); show[occ]=np.nan
    im=ax.imshow(show,cmap='viridis',origin='upper')
    ax.imshow(occ,cmap='gray_r',origin='upper',alpha=occ.astype(float)*0.85)
    ax.scatter(t[1],t[0],color='#E85D24',s=70,label='true')
    ax.set_title(f'step {i}, z={z:.2f}')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig('assets/09_markov_grid_localization.png',dpi=150,bbox_inches='tight'); plt.show()
print('true cell:', true)
print('MAP cell:', tuple(np.unravel_index(np.argmax(bel), bel.shape)))
print('belief entropy:', round(-np.nansum(bel*np.log(bel+1e-12)),3))

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Markov localization | 이산 상태공간 Bayes filter | Ch.7 Mobile Robot Localization |
| Histogram filter | grid 전체 belief 유지 | Ch.4 Nonparametric Filters |
| Entropy | localization uncertainty | active localization의 기준 |